# Phase 3: Building Search Indices (BM25 + FAISS)

This notebook:
- Loads product data
- Builds BM25 index for keyword search
- Generates embeddings using SentenceTransformers
- Builds FAISS index for semantic search
- Saves indices for later use

## 3.1 Imports and Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import pickle
import json
import time
from pathlib import Path
from tqdm import tqdm

# BM25
from rank_bm25 import BM25Okapi

# Embeddings
from sentence_transformers import SentenceTransformer

# FAISS
import faiss

print("✓ Imports successful")

✓ Imports successful


## 3.2 Load Product Data

In [2]:
# Load products
df_products = pd.read_parquet('../data/processed/products.parquet')

print(f"✓ Loaded {len(df_products):,} products")
print(f"\nColumns: {list(df_products.columns)}")
print(f"\nSample product:")
print(df_products.iloc[0].to_dict())

✓ Loaded 982,641 products

Columns: ['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale', 'product_text']

Sample product:
{'product_id': 'B000MOO21W', 'product_title': 'Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan', 'product_description': None, 'product_bullet_point': 'WhisperCeiling fans feature a totally enclosed condenser motor and a double-tapered, dolphin-shaped bladed blower wheel to quietly move air\nDesigned to give you continuous, trouble-free operation for many years thanks in part to its high-quality components and permanently lubricated motors which wear at a slower pace\nDetachable adaptors, firmly secured duct ends, adjustable mounting brackets (up to 26-in), fan/motor units that detach easily from the housing and uncomplicated wiring all lend themselves to user-friendly installation\nThis Panasonic fan has a built-in damper to prevent backdraft, which helps to prevent outside a

## 3.3 Create Searchable Text

In [3]:
def create_product_text(row):
    """
    Combine product fields into searchable text.
    Format: "title - brand | category | description"
    """
    parts = []
    
    # Title (most important)
    if pd.notna(row.get('product_title')):
        parts.append(str(row['product_title']))
    
    # Brand
    if pd.notna(row.get('product_brand')):
        parts.append(f"Brand: {row['product_brand']}")
    
    # Category/Class
    if pd.notna(row.get('product_class')):
        parts.append(f"Category: {row['product_class']}")
    
    # Description (truncate if too long)
    if pd.notna(row.get('product_description')):
        desc = str(row['product_description'])[:500]
        parts.append(desc)
    
    # Bullet points
    if pd.notna(row.get('product_bullet_point')):
        bullets = str(row['product_bullet_point'])[:300]
        parts.append(bullets)
    
    return " | ".join(parts)

In [4]:
# Create searchable text for each product
print("Creating searchable text for products...")
df_products['search_text'] = df_products.apply(create_product_text, axis=1)

print(f"✓ Created search text")
print(f"\nExample:")
print(df_products['search_text'].iloc[0][:500] + "...")

Creating searchable text for products...
✓ Created search text

Example:
Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan | Brand: Panasonic | WhisperCeiling fans feature a totally enclosed condenser motor and a double-tapered, dolphin-shaped bladed blower wheel to quietly move air
Designed to give you continuous, trouble-free operation for many years thanks in part to its high-quality components and permanently lubricated motors which wea...


## 3.4 Build BM25 Index

In [5]:
def tokenize(text):
    """Simple tokenizer: lowercase and split on whitespace/punctuation."""
    import re
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

In [6]:
print("Building BM25 index...")
start_time = time.time()

# Tokenize all documents
corpus = df_products['search_text'].tolist()
tokenized_corpus = [tokenize(doc) for doc in tqdm(corpus, desc="Tokenizing")]

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

build_time = time.time() - start_time
print(f"✓ BM25 index built in {build_time:.1f}s")
print(f"  Documents: {len(tokenized_corpus):,}")
print(f"  Avg tokens/doc: {np.mean([len(t) for t in tokenized_corpus]):.0f}")

Building BM25 index...


Tokenizing: 100%|██████████| 982641/982641 [00:23<00:00, 42461.27it/s]


✓ BM25 index built in 42.2s
  Documents: 982,641
  Avg tokens/doc: 94


## 3.5 Test BM25 Search

In [7]:
def bm25_search(query, top_k=5):
    """Search using BM25."""
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'rank': len(results) + 1,
            'product_id': df_products.iloc[idx]['product_id'],
            'title': df_products.iloc[idx]['product_title'],
            'score': scores[idx]
        })
    return results

In [8]:
# Test BM25 search
print("Testing BM25 search:")
print("-" * 50)

test_queries = ["ceramic mugs", "running shoes nike", "organic candles"]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = bm25_search(query, top_k=3)
    for r in results:
        print(f"  {r['rank']}. {r['title'][:60]}... (score: {r['score']:.2f})")

Testing BM25 search:
--------------------------------------------------

Query: 'ceramic mugs'
  1. Cute Marshmallow Shaped Hot Chocolate Mugs-Ceramic-Set of 4... (score: 19.55)
  2. Serami Classic Cream White Diner Mugs for Coffee with 11oz C... (score: 19.35)
  3. Only the Best Grandpas Get Promoted to Great Grandpa Ceramic... (score: 19.30)

Query: 'running shoes nike'
  1. Nike Women's Renew Run Running Shoes (Black/Pink/Orange, Num... (score: 26.46)
  2. Nike Womens Renew Run Chunky Workout Walking Shoes Black 7 M... (score: 26.29)
  3. Nike Women's Run Swift Running Shoes (White/White-Pure Plati... (score: 26.24)

Query: 'organic candles'
  1. Natural Light 12 Organic Beeswax Taper Candles 8" Tall X 5/8... (score: 18.28)
  2. Natural Light 6 Organic Beeswax Taper Candles 8" Tall X 5/8"... (score: 18.24)
  3. AIRA Soy Candles - Organic, Kosher, Vegan, in Mason Jar w/ T... (score: 17.53)


## 3.6 Generate Embeddings

In [9]:
# Load embedding model
print("Loading embedding model...")
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(model_name)

print(f"✓ Model loaded: {model_name}")
print(f"  Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

Loading embedding model...
✓ Model loaded: sentence-transformers/all-MiniLM-L6-v2
  Embedding dimension: 384


In [ ]:
# Generate embeddings for all products
print("\nGenerating embeddings for products...")
print("(This may take a few minutes)")

start_time = time.time()

# Use product titles for embeddings (faster and often sufficient)
texts_to_embed = df_products['product_title'].fillna('').tolist()

# Generate embeddings in batches
batch_size = 256
embeddings = embedding_model.encode(
    texts_to_embed,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True
)

embed_time = time.time() - start_time
print(f"\n✓ Embeddings generated in {embed_time:.1f}s")
print(f"  Shape: {embeddings.shape}")
print(f"  Dtype: {embeddings.dtype}")


Generating embeddings for products...
(This may take a few minutes)


Batches:   0%|          | 0/3839 [00:00<?, ?it/s]


✓ Embeddings generated in 2892.5s
  Shape: (982641, 384)
  Dtype: float32


: 

## 3.7 Build FAISS Index

In [ ]:
print("Building FAISS index...")
start_time = time.time()

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# Create index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product (cosine after normalization)

# Add vectors
index.add(embeddings)

build_time = time.time() - start_time
print(f"✓ FAISS index built in {build_time:.1f}s")
print(f"  Vectors: {index.ntotal:,}")
print(f"  Dimension: {dimension}")

Building FAISS index...


## 3.8 Test FAISS Search

In [ ]:
def faiss_search(query, top_k=5):
    """Search using FAISS (semantic similarity)."""
    # Encode query
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for i, (idx, score) in enumerate(zip(indices[0], scores[0])):
        results.append({
            'rank': i + 1,
            'product_id': df_products.iloc[idx]['product_id'],
            'title': df_products.iloc[idx]['product_title'],
            'score': float(score)
        })
    return results

In [ ]:
# Test FAISS search
print("Testing FAISS search:")
print("-" * 50)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = faiss_search(query, top_k=3)
    for r in results:
        print(f"  {r['rank']}. {r['title'][:60]}... (score: {r['score']:.3f})")

## 3.9 Compare BM25 vs FAISS

In [ ]:
print("BM25 vs FAISS Comparison:")
print("=" * 70)

comparison_queries = [
    "coffee cups",          # Synonym test (cups vs mugs)
    "eco friendly bags",    # Concept test
    "nike sneakers",        # Brand + synonym
]

for query in comparison_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    
    bm25_results = bm25_search(query, top_k=3)
    faiss_results = faiss_search(query, top_k=3)
    
    print("BM25 (keyword):")
    for r in bm25_results:
        print(f"  {r['rank']}. {r['title'][:55]}...")
    
    print("\nFAISS (semantic):")
    for r in faiss_results:
        print(f"  {r['rank']}. {r['title'][:55]}...")

## 3.10 Save Indices

In [ ]:
# Create indices directory
indices_dir = Path('../data/indices')
indices_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Save BM25 index
print("Saving BM25 index...")
bm25_data = {
    'bm25': bm25,
    'tokenized_corpus': tokenized_corpus
}
with open(indices_dir / 'bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_data, f)
print(f"✓ Saved: {indices_dir / 'bm25_index.pkl'}")

In [ ]:
# Save FAISS index
print("Saving FAISS index...")
faiss.write_index(index, str(indices_dir / 'faiss_index.bin'))
print(f"✓ Saved: {indices_dir / 'faiss_index.bin'}")

In [ ]:
# Save embeddings (for potential reuse)
print("Saving embeddings...")
np.save(indices_dir / 'product_embeddings.npy', embeddings)
print(f"✓ Saved: {indices_dir / 'product_embeddings.npy'}")

In [ ]:
# Save product ID mapping
print("Saving product ID mapping...")
product_ids = df_products['product_id'].tolist()
with open(indices_dir / 'product_ids.json', 'w') as f:
    json.dump(product_ids, f)
print(f"✓ Saved: {indices_dir / 'product_ids.json'}")

## 3.11 Summary

In [ ]:
print("\n" + "=" * 60)
print("PHASE 3 COMPLETE")
print("=" * 60)

print(f"""
Search Indices Summary
──────────────────────
Products indexed: {len(df_products):,}

BM25 Index (Keyword Search):
  - Documents: {len(tokenized_corpus):,}
  - Avg tokens/doc: {np.mean([len(t) for t in tokenized_corpus]):.0f}
  - File: data/indices/bm25_index.pkl

FAISS Index (Semantic Search):
  - Vectors: {index.ntotal:,}
  - Dimension: {dimension}
  - Model: {model_name}
  - File: data/indices/faiss_index.bin

Saved Files:
  - data/indices/bm25_index.pkl
  - data/indices/faiss_index.bin
  - data/indices/product_embeddings.npy
  - data/indices/product_ids.json

Next: Run 04_retrieval.ipynb (Hybrid search implementation)
""")